In [ ]:
# FIX CRITICO: Python 3.14 mudou o default de multiprocessing para 'forkserver'
# Isso causa OSError: Cannot allocate memory no WSL2 com num_workers > 0
# Forcar 'fork' resolve definitivamente o problema
import multiprocessing
multiprocessing.set_start_method('fork', force=True)
print(f'Metodo de multiprocessing: {multiprocessing.get_start_method()}')


In [ ]:
import os
import sys
import platform

print(f"Python version: {sys.version}")
print(f"OS: {platform.platform()}")
print(f"CPU Cores: {os.cpu_count()}")

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Habilitar o modo de alta performance (TF32) nas arquiteturas Ampere/Hopper
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True # Otimiza convs para tamanhos estáticos

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True  # Desativado para não perder performance extrema

set_seed(42)
print("Global seed set to 42.")

In [ ]:
# Diretório local para salvar o CIFAR-100
DATA_PATH = "./data"
print("Usando dataset em: " + DATA_PATH)


In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch

# Normalizacao padrao para CIFAR-100
mean = (0.5071, 0.4867, 0.4408)
std  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

DATA_PATH = "."
train_dataset = torchvision.datasets.CIFAR100(root=DATA_PATH, train=True, download=True, transform=transform_train)
test_dataset  = torchvision.datasets.CIFAR100(root=DATA_PATH, train=False, download=True, transform=transform_val)

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")


In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 6  # Aumentado: 12 cores disponíveis

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=4  # Aumentado: imagens 32x32 são leves
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4
)

print(f"DataLoaders otimizados criados com Batch Size = {BATCH_SIZE} e Workers = {NUM_WORKERS}")


In [ ]:
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

In [ ]:
# ==============================================================================
# OTIMIZADOR SAM (Sharpness-Aware Minimization)
# Adicionado para a Fase 3: Melhora generalização escapando de mínimos agudos.
# ==============================================================================
import torch

class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)

        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)  # climb to the local maximum "w + e(w)"
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  # get back to "w" from "w + e(w)"
        # O base_optimizer.step() original foi REMOVIDO daqui.
        # Ele sera chamado diretamente pelo GradScaler do PyTorch no loop de treino (AMP).
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        pass # Nao usado neste design customizado com AMP

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
                    torch.stack([
                        ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        return norm


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
# === FASE 3: SAM + SWA (Mínimos Planos + Média de Pesos) ===
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

EPOCHS = 200
LEARNING_RATE = 0.10

# SAM encapsulando o SGD base
base_optimizer = torch.optim.SGD
optimizer = SAM(
    model.parameters(), base_optimizer, rho=0.05,
    lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4, nesterov=True
)

# CosineAnnealing puro, épocas 1-149
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# === SWA (Stochastic Weight Averaging) ===
# Ativa a partir de 75% do treino (época 150/200)
# swa_lr = 0.01 (abaixo do LR na transição ~0.0147 — sem salto brusco)
SWA_START = 150
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=0.01, anneal_epochs=10)
# anneal_epochs=10: LR desce suavemente de 0.0147 até 0.01 em 10 épocas (sem choque)

print(f"SAM+SWA | {EPOCHS} épocas | LR inicial: {LEARNING_RATE}")
print(f"SWA ativa da época {SWA_START} | SWA LR final: 0.01 (transição suave)")


In [ ]:
import time
from torch.amp import autocast, GradScaler
from torchmetrics import Accuracy

scaler = GradScaler('cuda')
accuracy_metric = Accuracy(task="multiclass", num_classes=100).to(device)
top5_accuracy_metric = Accuracy(task="multiclass", num_classes=100, top_k=5).to(device)

best_acc = 0.0
cutmix_prob = 0.5

print("Iniciando Treinamento SAM + SWA...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    start_time = time.time()
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        inputs = inputs.to(memory_format=torch.channels_last)

        # CutMix
        r = np.random.rand(1)
        if r < cutmix_prob:
            lam = np.random.beta(1.0, 1.0)
            rand_index = torch.randperm(inputs.size()[0]).cuda()
            target_a = targets
            target_b = targets[rand_index]
            bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
            inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size()[-1] * inputs.size()[-2]))

            # SAM PASSO 1
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
            scaler.scale(loss).backward()
            optimizer.first_step(zero_grad=True)

            # SAM PASSO 2
            with autocast('cuda'):
                outputs_adv = model(inputs)
                loss_adv = criterion(outputs_adv, target_a) * lam + criterion(outputs_adv, target_b) * (1. - lam)
            scaler.scale(loss_adv).backward()
            optimizer.second_step(zero_grad=True)
            scaler.step(optimizer.base_optimizer)
            scaler.update()

        else:
            # SAM PASSO 1
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            optimizer.first_step(zero_grad=True)

            # SAM PASSO 2
            with autocast('cuda'):
                outputs_adv = model(inputs)
                loss_adv = criterion(outputs_adv, targets)
            scaler.scale(loss_adv).backward()
            optimizer.second_step(zero_grad=True)
            scaler.step(optimizer.base_optimizer)
            scaler.update()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    # --- Validação com modelo SGD/SAM normal ---
    model.eval()
    val_loss = 0
    accuracy_metric.reset()
    top5_accuracy_metric.reset()

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.to(memory_format=torch.channels_last)
            with autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            val_loss += loss.item()
            accuracy_metric.update(outputs, targets)
            top5_accuracy_metric.update(outputs, targets)

    val_acc = accuracy_metric.compute().item() * 100
    val_top5 = top5_accuracy_metric.compute().item() * 100

    # === SWA: Atualiza média de pesos a partir da época SWA_START ===
    if epoch + 1 >= SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()          # LR constante do SWA
    else:
        scheduler.step()              # CosineAnnealing nas épocas iniciais

    epoch_time = time.time() - start_time
    current_lr = optimizer.base_optimizer.param_groups[0]['lr']

    print(f'Epoch [{epoch+1}/{EPOCHS}] | Time: {epoch_time:.0f}s | '
          f'Loss: {train_loss/len(train_loader):.4f} | '
          f'Val Loss: {val_loss/len(test_loader):.4f} | '
          f'Val Acc: {val_acc:.2f}% | Val Top5: {val_top5:.2f}% | '
          f'LR: {current_lr:.5f}' +
          (' [SWA ATIVO]' if epoch + 1 >= SWA_START else ''))

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_wrn28_10_cifar100.pth')
        print(f'>>> Melhor modelo salvo! Acurácia: {best_acc:.2f}%')

# === PÓS-TREINO: Calibra o BatchNorm do SWA e avalia o modelo médio ===
print("\n=== Calibrando BatchNorm do SWA (1 passada no dataset de treino)... ===")
torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)

swa_model.eval()
accuracy_metric.reset()
top5_accuracy_metric.reset()

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        inputs = inputs.to(memory_format=torch.channels_last)
        with autocast('cuda'):
            outputs = swa_model(inputs)
        accuracy_metric.update(outputs, targets)
        top5_accuracy_metric.update(outputs, targets)

swa_acc = accuracy_metric.compute().item() * 100
swa_top5 = top5_accuracy_metric.compute().item() * 100

print(f"\n{'='*60}")
print(f"RESULTADO FINAL SAM (melhor snapshot):  {best_acc:.2f}%")
print(f"RESULTADO FINAL SWA (média de pesos):   {swa_acc:.2f}% | Top5: {swa_top5:.2f}%")
print(f"{'='*60}")

# Salva o modelo SWA se for melhor
if swa_acc > best_acc:
    torch.save(swa_model.state_dict(), 'best_wrn28_10_cifar100_SWA.pth')
    print(f">>> Modelo SWA é SUPERIOR! Salvo como best_wrn28_10_cifar100_SWA.pth")
else:
    print(f">>> Modelo SAM snapshot foi superior. SWA não superou desta vez.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix

# Carregar o melhor modelo
model.load_state_dict(torch.load('best_wrn28_10_cifar100.pth'))
model.eval()

confmat = ConfusionMatrix(task="multiclass", num_classes=100).to(device)
accuracy_metric.reset()
top5_accuracy_metric.reset()

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        confmat.update(outputs, targets)
        accuracy_metric.update(outputs, targets)
        top5_accuracy_metric.update(outputs, targets)

cm = confmat.compute().cpu().numpy()
final_acc = accuracy_metric.compute().item() * 100
final_top5 = top5_accuracy_metric.compute().item() * 100

print(f"\n--- RESULTADOS FINAIS ---")
print(f"Melhor Top-1 Accuracy: {final_acc:.2f}%")
print(f"Melhor Top-5 Accuracy: {final_top5:.2f}%")

plt.figure(figsize=(20, 20))
sns.heatmap(cm, cmap='Blues', annot=False, xticklabels=False, yticklabels=False)
plt.title('Confusion Matrix - CIFAR-100', fontsize=16)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()